# > Initial setup

In [2]:
AI_GENERATED = """
Chart 1: Scoreboard Overview
L2: Sales £733,215, profit £93,439, returns 4,655 units, quantity 12,476.
L3: Sales/quantity grow YoY but returns rise too.
L4: Monitor returns to protect margins amid expansion.
Chart 2: 2023 | Sales vs Targets
L2: Sales hit £733K, exceeding the target of £715K by +20.4% vs PY.
L3: All categories meet or exceed targets except for Phones and Binders.
L4: Focus on maintaining consistency in growth across all segments.
Chart 3: Segment | Sales vs Targets
L2: Consumer sales are down -£24.3K, Corporate -£6.7K, Home Office +£33.2K.
L3: Home Office shows strong performance.
L4: Consider increasing marketing efforts for the Home Office segment.
Chart 4: Top 5 Sub-Categories | Sales vs Targets
L2: Phones lead with £10.6K sales, followed by Chairs at £5.1K and Binders at £0.9K.
L3: The top three sub-categories are all in positive growth.
L4: Continue to invest in product development for the most profitable categories.
Chart 5: Sales by Location | Top 5 States
L2: California leads with £146,388, New York at £93,923, Washington at £65,540, Texas at £43,422, and Pennsylvania at £42,688.
L3: The top states are concentrated in the West.
L4: Target expansion into high-value markets like California.
Chart 6: Top 5 Manufacturers | Sales vs Targets
L2: Canon leads with +£6.1K sales, Global -£3.9K, Hon -£11.3K, GBC +£6.8K, Fellowes +£9.8K.
L3: The top three are all in positive growth.
L4: Focus on maintaining strong performance from the top manufacturers.
"""

In [3]:
HUMAN_GENERATED = """
Chart 1: Scoreboard overview
L2: Sales are about 733K, profit is 93K, returns are around 4.7K, and quantity is about 12.5K units, each showing a noticeable yoy change.
L3: Overall, the business is growing in sales and quantity with generally improving profit.
L4: However, while the company is successfully expanding revenue and volume, the return rates should be monitored to ensure growth does not erode profitability or customer satisfaction.

Chart 2: 2023 | Sales vs Targets
L2: Sales started at around 40K in January and ended the year at around 80K, but it did not reach the target.
L3: The chart shows green circles for January, June, August, October, and November, which reached the target.
L4: In conclusion, sales improved over time, with high‑performing later months helping lift weaker early‑year periods in the next cycle.

Chart 3: Segment | Sales vs Targets
L2: Consumer and Corporate segments sit below their targets.
L3: The Home Office exceeds its target by 33.2K.
L4: Strategy should probe what is driving Home Office success and apply its tactics to Consumer and Corporate segments to close their gaps.

Chart 4: Top 5 Sub-Categories | Sales vs Targets
L2: Phones lead the sub-categories sales with +10.6K sales
L3: Phones, binders, and copiers have exceeded their targets, while the chairs lagged behind by 5.1K, and storage was just 0.9K below its target.
L4: The target should be adjusted for chairs as it is higher than the phones' target, which phones are hot commodity in these days

Chart 5: Sales by Location | Top 5 States
L2: California leads the sales by 146K, New York, Washington, Texas, and Pennsylvania follow with progressively smaller yet still substantial figures.
L3: A small group of states contributes a disproportionate share of total sales, with coastal and large‑population states dominating the top five.
L4: This geographic condition suggests expansion into mid-tier states could be the next growth target.

Chart 6: Top 5 Manufacturers chart | Sales vs Targets
L2: Canon leads with +6.1K sales
L3: Canon, GBC, and Fellowes have met their targets, while the Global and Hon manufacturers have lagged behind theirs.
L4: Even though the Global and Hon have their sales ranked in 2nd and 3rd place, the target is too high, should lower the target
"""

# > Evaluation

## 1. ROUGE

In [4]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"], use_stemmer=True
)
scores = scorer.score(HUMAN_GENERATED, AI_GENERATED)

print("ROUGE SCORES")
print("=" * 40)
for metric, result in scores.items():
    print(f"\n{metric.upper()}")
    print(f"  Precision : {result.precision:.4f}")
    print(f"  Recall    : {result.recall:.4f}")
    print(f"  F1        : {result.fmeasure:.4f}")

ROUGE SCORES

ROUGE1
  Precision : 0.6409
  Recall    : 0.4323
  F1        : 0.5163

ROUGE2
  Precision : 0.2829
  Recall    : 0.1906
  F1        : 0.2278

ROUGEL
  Precision : 0.4517
  Recall    : 0.3047
  F1        : 0.3639


## 2. BERTScore

In [ ]:
import torch
from bert_score import score as bert_score

# ── Force Apple Silicon GPU (MPS) ───────────────────────────
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

P, R, F1 = bert_score(
    [AI_GENERATED],
    [HUMAN_GENERATED],
    lang="en",
    model_type="distilbert-base-uncased",
    num_layers=5,
    device=device,
    verbose=True
)

print("\nBERTScore")
print("=" * 40)
print(f"  Precision : {P[0].item():.4f}")
print(f"  Recall    : {R[0].item():.4f}")
print(f"  F1        : {F1[0].item():.4f}")

Using device: mps


## 3. G-Eval (LLM-as-Judge)

In [ ]:
# G-Eval scores 4 dimensions (1–5 scale each):
#   Factual Accuracy  – do the numbers match the reference?
#   Completeness      – are all charts covered?
#   Analytical Depth  – does L3/L4 go beyond surface-level facts?
#   Conciseness       – is language crisp and executive-appropriate?

import os, json
from dotenv import load_dotenv
import anthropic

load_dotenv(os.path.expanduser("~/.env"))

GEVAL_PROMPT = """You are an expert evaluator for BI dashboard insight quality.

You will be given:
- REFERENCE: a human-written expert insight (ground truth)
- CANDIDATE: an AI-generated insight to evaluate

Score the CANDIDATE on these 4 dimensions, each from 1 (poor) to 5 (excellent):
1. Factual Accuracy: Do the numbers, signs, and comparisons match the REFERENCE?
2. Completeness: Does the CANDIDATE cover all charts/sections in the REFERENCE?
3. Analytical Depth: Does the CANDIDATE identify patterns and implications (L3/L4), not just restate facts?
4. Conciseness: Is the language crisp and executive-appropriate, without padding or vague filler?

Respond ONLY with valid JSON in this exact format:
{{
  "factual_accuracy": <1-5>,
  "completeness": <1-5>,
  "analytical_depth": <1-5>,
  "conciseness": <1-5>,
  "overall": <average of the four, rounded to 2 decimal places>,
  "rationale": "<2-3 sentences explaining the scores>"
}}

REFERENCE:
{reference}

CANDIDATE:
{candidate}
"""

client = 
opic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

message = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": GEVAL_PROMPT.format(
            reference=HUMAN_GENERATED,
            candidate=AI_GENERATED
        )
    }]
)

result = json.loads(message.content[0].text.strip())

print("G-EVAL SCORES")
print("=" * 40)
print(f"  Factual Accuracy  : {result['factual_accuracy']} / 5")
print(f"  Completeness      : {result['completeness']} / 5")
print(f"  Analytical Depth  : {result['analytical_depth']} / 5")
print(f"  Conciseness       : {result['conciseness']} / 5")
print(f"  Overall           : {result['overall']} / 5")
print(f"\n  Rationale: {result['rationale']}")